# V2X ??? ??

In [ ]:
import numpy as np
import cv2
import torch
import pandas as pd
import re
import pickle as pkl
import json
import glob
import shutil
import os
import json
import pathlib as pl
from typing import List, Tuple, Dict, Union, Optional
from tqdm import trange, tqdm
from glob import glob
from collections import Counter
from sklearn.model_selection import train_test_split

from preprocessing import V2XPreprocessing
from modules.v2x import V2XData,V2XDataLabeled
from modules.metrics import *
from tsai.all import *
from tqdm import tqdm
from modules.RunTSAI import RunTSAI
from sklearn.utils import shuffle


data_path_8 = "/data/NIA50/50-1/data/v2xfinal/8?"
data_path_8s = "/data/NIA50/50-1/data/v2xfinal/8???"
data_path_9 = "/data/NIA50/50-1/data/v2xfinal/9?"
data_path_10 = "/data/NIA50/50-1/data/v2xfinal/10?"
data_path_11 = "/data/NIA50/50-1/data/v2xfinal/11?"

## CSV, JSON ?? ?? ??? ???
- CSV, JSON ??? ? ??? ??
  - 8?
  -  ? CSV
  -  ? JSON
  - 9?
  -  ? CSV
  -  ? JSON

In [ ]:
# 9?csv??? json??? ?? ???? ???
csv_files9, json_files9 = V2XPreprocessing.get_csv_data(data_path_9 + "/CSV"), \
                            V2XPreprocessing.get_json_data(data_path_9 + "/JSON")
print(f"csv file count: {len(csv_files9)}, json file count: {len(json_files9)}")

In [ ]:
# csv??? json?? ???? pickle? ??
with open("pickles/9?/CSV9.pkl", "wb") as f:
    pkl.dump(csv_files9, f)
with open("pickles/9?/JSON9.pkl", "wb") as f:
    pkl.dump(json_files9, f)

In [ ]:
# csv??? json?? ? ???? pickle? ???
with open("pickles/9?/CSV9.pkl", "rb") as f:
    csv_files9 = pkl.load(f)
with open("pickles/9?/JSON9.pkl", "rb") as f:
    json_files9 = pkl.load(f)

In [ ]:
# CSV?? concat, pickle? ??
df9csv = [pd.read_csv(f) for f in tqdm(csv_files9, desc="Reading csv files")]
df9csv = pd.concat(df9csv, ignore_index=True)
with open("pickles/9?/df9csv.pkl", "wb") as f:
    pkl.dump(df9csv, f)

In [ ]:
# CSV?? concat pickle? ???
with open("pickles/9?/df9csv.pkl", "rb") as f:
    df9csv = pkl.load(f)

In [ ]:
df9csv.head()

In [ ]:
# NaN? ?? ? NaN?? ??? ??
df9csv = df9csv.dropna()
# counts rows if there's NaN value
df9csv.isna().sum()

In [ ]:
df9csv.shape

In [ ]:
# int or float type? ?? ?? ??? ??
for col in df9csv.columns:
    if col in ["VEHICLE_ID", "VEHICLE_TYPE"]:
        continue
    # print if value is not float or int
    print(f"{col}: {df9csv[col].apply(lambda x: type(x) not in [float, int]).sum()}")

In [ ]:
# exclude ["ISSUE_DATE"] column, values to int on ["VEHICLE_CLASS", "HEADING"], str on ["VEHICLE_ID", "VEHICLE_TYPE"], float on others

for col in df9csv.columns:
    if col in ["ISSUE_DATE"]:
        df9csv.drop(col, axis=1, inplace=True)
    elif col in ["VEHICLE_CLASS", "HEADING"]:
        df9csv[col] = df9csv[col].astype(int)
    elif col in ["VEHICLE_ID", "VEHICLE_TYPE"]:
        df9csv[col] = df9csv[col].astype(str)
    else:
        df9csv[col] = df9csv[col].astype(float)

In [ ]:
# value counts of VEHICLE_ID VEHICLE_CLASS VEHICLE_TYPE
for col in df9csv.columns:
    if col in ["VEHICLE_ID", "VEHICLE_CLASS", "VEHICLE_TYPE"]:
        print(f"{col}: {df9csv[col].value_counts()}")
        print(f"{col} unique: {df9csv[col].unique().shape}", end="\n\n")

In [ ]:
# categorize ["VEHICLE_ID", "VEHICLE_CLASS", "VEHICLE_TYPE"] columns
from sklearn.preprocessing import LabelEncoder

for col in df9csv.columns:
    if col in ["VEHICLE_ID", "VEHICLE_CLASS", "VEHICLE_TYPE"]:
        encoder = LabelEncoder()
        encoder.fit(df9csv[col])
        categorized_array = encoder.transform(df9csv[col])
        print(categorized_array)
        df9csv[col] = categorized_array

In [ ]:
# index reset
df9csv.reset_index(drop=True, inplace=True)
df9csv

# Standarization

In [ ]:
# standardize
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
df9csv_scaled = scaler.fit_transform(df9csv)
df9csv_scaled = pd.DataFrame(df9csv_scaled, columns=df9csv.columns)
df9csv_scaled

# CSV ?? Dataframe pickle? ??

In [ ]:
with open("pickles/9?/df9_scaled.pkl", "wb") as f:
    pkl.dump(df9csv_scaled, f)

In [ ]:
df9csv_scaled = pd.read_pickle("pickles/9?/df9_scaled.pkl") # load pickle

In [ ]:
# reshape 1350000 rows ? 9 columns to 1350000 rows x 9 columns x 10, and changed to numpy array
df9csv_scaled_npy = df9csv_scaled.values.reshape(df9csv_scaled.shape[0]//10, 10, 9)
df9csv_scaled_npy = df9csv_scaled_npy.transpose(0, 2, 1)

In [ ]:
# save numpy array as npy file
np.save("pickles/9?/X_9.npy", df9csv_scaled_npy)

In [ ]:
print(df9csv_scaled_npy.shape)
df9csv_scaled_npy[:2]

----------------------

# JSON

In [ ]:
with open("pickles/9?/JSON9.pkl", "rb") as f:
    json_files9 = pkl.load(f)

In [ ]:
# json info
with open(json_files9[0], "r") as f:
    json_sample = json.load(f)
json_sample

In [ ]:
df9_json = pd.read_json(json_files9[0])
# df8_json = df8_json['Annotation'][['Turn', 'Lane', 'Speed', 'Hazard']].to_frame().T
# values = df8_json.values.tolist()[0]
# values
df9_json

In [ ]:
json9_list = []
cols = ['Turn', 'Lane', 'Speed', 'Hazard']
print(cols)

df8json = pd.DataFrame(columns=cols)
for i in trange(len(json_files9)):
    with open(json_files9[i], "r") as f:
        temp_json = json.load(f)
        temp_json = [temp_json['Annotation'][key] for key in temp_json['Annotation'].keys() if key in cols]
        json9_list.append(temp_json)
        
# json8_list = [x[:4] for x in json8_list]
json9_npy = np.array(json9_list)

In [ ]:
np.save("pickles/9?/y_9.npy", json9_npy)

In [ ]:
json9_npy = np.load("pickles/9?/y_9.npy")
json9_npy.shape

In [ ]:
json9_npy

-------------------------------------------

# Time series classification

In [ ]:
X = np.load("pickles/9?/X_9.npy")
y = np.load("pickles/9?/y_9.npy")

###
from collections import Counter
print("turn:", np.unique(y[:, 0]))
print("lane:", np.unique(y[:, 1]))
print("speed:", np.unique(y[:, 2]))
print("hazard:", np.unique(y[:, 3]))
print("turn:", Counter(y[:, 0]))
print("lane:", Counter(y[:, 1]))
print("speed:", Counter(y[:, 2]))
print("hazard:", Counter(y[:, 3]))
print(f"X shape: {X.shape}, y shape: {y.shape}")

In [ ]:
y[:10000]

In [ ]:
from tsai.all import *
config = AttrDict(
    batch_tfms = TSStandardize(by_sample=True),
    arch_config = {},
    architecture = LSTM, # LSTM, MLSTM_Plus
    lr = 1e-3,
    n_epochs = 200,
)

X = np.load("pickles/9?/X_9.npy")
y = np.load("pickles/9?/y_9.npy")
split_all = get_splits(y[:,0], shuffle=True, valid_size=0.2, test_size=0.2, random_state=42)
split = split_all

In [ ]:
test_eq(X.shape, (X.shape[0], 9, 10))
test_eq(y.shape, (X.shape[0], 4))

In [ ]:
tfms = [None, [MultiCategorize, OneHotEncode]]
dsets = TSDatasets(X, y, tfms=tfms, splits=split)
dls = TSDataLoaders.from_dsets(dsets.train, dsets.valid, bs=[64, 128], shuffle_train=False, batch_tfms=[TSStandardize()])

model = InceptionTime(dls.vars, dls.c)
learn = Learner(dls, model, loss_func=BCEWithLogitsLossFlat(), metrics=[accuracy_multi, precision_multi, F1_multi])
learn.fit_one_cycle(10)
learn.recorder.plot_loss()

In [ ]:
learn.recorder.plot_metrics()
learn.save_all("models/V2X_9")

-------------------

# Load model, get prediction

In [ ]:
learn = load_learner_all(path="V2X_9", dls_fname='dls', model_fname='model', learner_fname='learner')
dls = learn.dls
dls_valid = dls.valid
test_probas, test_targets, test_preds = learn.get_preds(dl=dls_valid, with_decoded=True)

In [ ]:
print(test_probas[0])
print(test_targets[0])
print(test_preds[0])

In [ ]:
# check test_targets[0] if it has two 1s
for att in test_targets:
    if np.count_nonzero(att) > 1:
        print(att, np.count_nonzero(att))